# Resume frozen-backbone boat training on a Colab GPU

Resume `boat_v4s_frozen` from its saved checkpoint, including optimizer state, epoch counter and learning-rate schedule. The original checkpoint stopped at epoch 42 of 80.

1. Choose a GPU under **Runtime > Change runtime type**.
2. Upload `boat_v4s_frozen_resume.zip` to Google Drive and update `ZIP_PATH` below.
3. Run the setup cells in order, then resume training.


In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Set ZIP_PATH to the bundle you uploaded. Extraction resets /content/work.
ZIP_PATH = '/content/drive/MyDrive/trainFreeze/boat_v4s_frozen_resume.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!find /content/work -maxdepth 5 -iname 'last.pt' -o -iname 'dataset.yaml'


In [ ]:
!pip install -q ultralytics==8.4.138


In [ ]:
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
import yaml
settings = yaml.safe_load(content)
settings['path'] = '/content/work/yolo_dataset_v4'
new_content = yaml.safe_dump(settings, sort_keys=False)
yaml_path.write_text(new_content)
print(yaml_path.read_text())


In [ ]:
# Adapt a trusted checkpoint and its training arguments to Colab before resuming.
import torch, glob

ckpt_path = glob.glob('/content/work/**/boat_v4s_frozen/weights/last.pt', recursive=True)[0]
print('checkpoint:', ckpt_path)
run_dir = str(pathlib.Path(ckpt_path).parent.parent)
print('run_dir:', run_dir)

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
ta = ckpt['train_args']
print('--- before ---')
print({k: ta[k] for k in ('device', 'data', 'save_dir', 'project', 'name')})

ta['device'] = 0
ta['data'] = '/content/work/yolo_dataset_v4/dataset.yaml'
ta['save_dir'] = run_dir
ta['workers'] = 2  # Keep CPU worker usage modest.
ckpt['train_args'] = ta
torch.save(ckpt, ckpt_path)

import yaml
args_yaml_path = pathlib.Path(run_dir) / 'args.yaml'
with open(args_yaml_path) as f:
    y = yaml.safe_load(f)
y['device'] = 0
y['data'] = '/content/work/yolo_dataset_v4/dataset.yaml'
y['save_dir'] = run_dir
y['workers'] = 2
with open(args_yaml_path, 'w') as f:
    yaml.safe_dump(y, f)

print('--- after ---')
print({k: ta[k] for k in ('device', 'data', 'save_dir', 'project', 'name')})


In [ ]:
# Resume from the latest checkpoint; retain optimizer and scheduler state.
from ultralytics import YOLO

model = YOLO(ckpt_path)
results = model.train(resume=True)


In [ ]:
# If a session disconnects, restore the latest checkpoint from Drive before resuming.


In [ ]:
!mkdir -p /content/drive/MyDrive/boat_v4s_frozen_results
!cp -r "$run_dir" /content/drive/MyDrive/boat_v4s_frozen_results/
print('Copied to: Google Drive > boat_v4s_frozen_results')


## Retrieve and evaluate the trained checkpoint

Download the result folder from the Google Drive destination printed above. Keep `weights/best.pt`, `weights/last.pt`, `args.yaml`, and `results.csv` together so checkpoint provenance is available. Pass the exact `best.pt` path to `run.py --yolo-weights` and evaluate it on independently reviewed recordings.

Colab sessions can disconnect. Keep recent checkpoints in Drive and resume from the latest `last.pt`; do not restart from the initial bundle and assume it contains newer training progress. Session duration and training speed are not guaranteed.
